# HR Analytics – Predict Employee Attrition
## Phase 3: Exploratory Data Analysis (EDA)

**Objective:** Turn cleaned data into business understanding. EDA is not about making charts for the sake of it — each visualization should answer an HR question: *Who is leaving, where, and under what conditions?*

**Dataset:** `data/processed/hr_cleaned.csv` (post Phase 2 preprocessing)

---

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMAGES_DIR = PROJECT_ROOT / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from eda_helpers import plot_attrition_rate, plot_ordinal_attrition, attrition_rate_table

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "hr_cleaned.csv")
OVERALL_RATE = df["Attrition_Flag"].mean() * 100

print(f"Records: {len(df):,} | Overall attrition rate: {OVERALL_RATE:.1f}%")
print(f"Leavers: {df['Attrition_Flag'].sum()} | Retained: {(df['Attrition_Flag']==0).sum()}")

---
## 1. Employee Attrition Distribution

**Why:** Before drilling into drivers, we establish the baseline — how imbalanced is the target, and what does the class split look like for modeling?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

attrition_counts = df["Attrition"].value_counts()
colors = ["#2ecc71", "#e74c3c"]
axes[0].bar(attrition_counts.index, attrition_counts.values, color=colors)
axes[0].set_title("Employee Count by Attrition Status")
axes[0].set_ylabel("Number of Employees")
for i, v in enumerate(attrition_counts.values):
    axes[0].text(i, v + 15, str(v), ha="center", fontweight="bold")

pie_labels = [f"{label} ({count/len(df)*100:.1f}%)" for label, count in attrition_counts.items()]
axes[1].pie(attrition_counts, labels=pie_labels, colors=colors, startangle=90)
axes[1].set_title("Attrition Proportion")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_01_attrition_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

**Business Insights:**
1. **16.1% overall attrition** — roughly 1 in 6 employees left. That is 237 exits in a 1,470-person workforce — material enough to warrant targeted retention investment.
2. The dataset is **class-imbalanced (84% vs 16%)**. Accuracy alone would be misleading in Phase 5; we should prioritize Recall and F1 for identifying leavers.
3. Attrition is the **minority class but not rare** — 237 "Yes" records provide a stable foundation for modeling.

**HR Implications:** Benchmark 16% against industry norms. Even if acceptable, replacing 237 employees carries significant recruiting and productivity costs — a data-driven retention program is justified.

---
## 2. Department-wise Attrition

**Why:** Which organizational units are losing people fastest?

In [ ]:
summary_2 = plot_attrition_rate(
    df, "Department", "Department-wise Attrition",
    horizontal=False,
    save_path=IMAGES_DIR / "eda_02_department.png",
)
summary_2

**Business Insights:**
1. **Sales attrition is 20.6%** — highest among major departments, likely driven by target pressure and external poaching.
2. **R&D attrition is 13.8%** — lowest among large departments; technical career paths may be retaining staff better.
3. **HR shows 19.0% attrition** on a small base (63 employees) — each exit disrupts core people operations.

**HR Implications:** Launch a Sales retention task force first. Document and replicate R&D engagement practices company-wide.

---
## 3. Job Role vs Attrition

**Why:** Role-level granularity reveals hotspots that department averages hide.

In [ ]:
summary_3 = plot_attrition_rate(
    df, "JobRole", "Job Role vs Attrition",
    horizontal=True,
    save_path=IMAGES_DIR / "eda_03_jobrole.png",
)
summary_3

**Business Insights:**
1. **Sales Representative attrition is 39.8%** — nearly 2.5× the company average; the clearest red flag in the dataset.
2. **Laboratory Technician (23.9%) and Human Resources (23.1%)** also show elevated turnover — often entry-level roles with limited upward mobility.
3. **Research Director (2.5%) and Manager (4.9%)** have the lowest attrition — seniority and compensation appear protective.

**HR Implications:** Redesign Sales Rep onboarding, quotas, and manager check-ins. Build promotion pathways for Lab Technicians.

---
## 5. Salary Band vs Attrition

**Why:** Pay bands translate raw income into policy-relevant segments.

In [ ]:
summary_5 = plot_attrition_rate(
    df, "Salary_Band", "Salary Band vs Attrition",
    horizontal=False,
    save_path=IMAGES_DIR / "eda_05_salary_band.png",
)
summary_5

**Business Insights:**
1. **Low band attrition is 29.3%** — almost triple the Very High band (10.3%). Underpayment is a tangible driver of exits.
2. **High and Very High bands (~10–11%)** show similar, lower turnover — competitive pay retains senior staff.
3. **Medium band (14.2%)** sits in the middle — employees may leave for modest external offers if raises stall.

**HR Implications:** Audit Low-band compensation against market rates. Consider retention bonuses for high performers stuck in lower bands.

---
## 9. Gender vs Attrition

**Why:** Gender differences should be monitored for equity, not assumed as causes.

In [ ]:
summary_9 = plot_attrition_rate(
    df, "Gender", "Gender vs Attrition",
    horizontal=False,
    save_path=IMAGES_DIR / "eda_09_gender.png",
)
summary_9

**Business Insights:**
1. **Male attrition (17.0%) exceeds Female (14.8%)** — a modest gap, not dramatic but worth monitoring across roles.
2. Gender alone does not explain attrition; interaction with role, pay, and overtime matters more.
3. Sample is **60% Male, 40% Female** — representative enough for segment comparison.

**HR Implications:** Analyze gender × department × pay interactions in Phase 8. Ensure retention programs are equitable, not one-size-fits-all.

---
## 10. Education Field vs Attrition

**Why:** Background may correlate with role fit and alternative job options.

In [ ]:
summary_10 = plot_attrition_rate(
    df, "EducationField", "Education Field vs Attrition",
    horizontal=True,
    save_path=IMAGES_DIR / "eda_10_education_field.png",
)
summary_10

**Business Insights:**
1. **Technical Degree (24.2%) and Human Resources field (25.9%)** show the highest attrition among education backgrounds.
2. **Medical (13.6%) and Other (13.4%)** are closest to company average — more stable retention profiles.
3. Marketing (22.0%) is elevated — may reflect competitive external market for marketing talent.

**HR Implications:** For Technical Degree holders in R&D/Sales, focus on project ownership and skill development to reduce poaching vulnerability.

---
## 18. Business Travel vs Attrition

**Why:** Travel frequency is a proxy for role intensity and work-life strain.

In [ ]:
summary_18 = plot_attrition_rate(
    df, "BusinessTravel", "Business Travel vs Attrition",
    horizontal=False,
    save_path=IMAGES_DIR / "eda_18_business_travel.png",
)
summary_18

**Business Insights:**
1. **Frequent travelers attrition is 24.9%** — more than 3× Non-Travel employees (8.0%).
2. **Travel_Rarely (15.0%)** aligns with company average — occasional travel is manageable.
3. Travel-heavy roles compound with overtime and satisfaction scores to increase burnout risk.

**HR Implications:** Offer travel perks, flexible recovery days, or rotation for frequent travelers. Review whether all frequent travel is truly necessary.

---
## 19. Marital Status vs Attrition

**Why:** Life stage influences mobility, risk tolerance, and benefit needs.

In [ ]:
summary_19 = plot_attrition_rate(
    df, "MaritalStatus", "Marital Status vs Attrition",
    horizontal=False,
    save_path=IMAGES_DIR / "eda_19_marital_status.png",
)
summary_19

**Business Insights:**
1. **Single employees attrition is 25.5%** — highest among marital groups; fewer geographic/financial anchors.
2. **Divorced employees (10.1%)** show the lowest attrition — possibly prioritizing job stability.
3. **Married employees (12.5%)** fall below company average — family considerations may reduce job switching.

**HR Implications:** Tailor benefits communication for singles (growth, mobility) vs. married employees (stability, family benefits). Avoid stereotyping — use as segmentation hint only.

---
## 20. Overtime vs Attrition

**Why:** Overtime is one of the strongest operational levers HR can control.

In [ ]:
summary_20 = plot_attrition_rate(
    df, "OverTime", "Overtime vs Attrition",
    horizontal=False,
    save_path=IMAGES_DIR / "eda_20_overtime.png",
)
summary_20

**Business Insights:**
1. **Overtime workers attrition is 30.5%** vs **10.4% for non-overtime** — a 20-point gap and one of the sharpest splits in the dataset.
2. 416 employees (28%) work overtime — a large at-risk population.
3. Overtime likely interacts with low pay bands and poor work-life balance to accelerate burnout.

**HR Implications:** Cap overtime in high-risk teams, hire to reduce overload, and compensate OT fairly. This is a high-impact, actionable finding.

---
## 4. Monthly Income vs Attrition

**Why:** Compensation is often the first hypothesis in turnover analysis. Comparing income distributions between stayers and leavers tests whether exits are financially driven.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="Attrition", y="MonthlyIncome", hue="Attrition", palette=["#2ecc71", "#e74c3c"], legend=False, ax=axes[0])
axes[0].set_title("Monthly Income by Attrition Status")
axes[0].set_ylabel("Monthly Income ($)")

sns.kdeplot(data=df, x="MonthlyIncome", hue="Attrition", fill=True, alpha=0.4, palette=["#2ecc71", "#e74c3c"], ax=axes[1])
axes[1].set_title("Income Distribution — Stayers vs Leavers")
axes[1].set_xlabel("Monthly Income ($)")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_04_monthly_income.png", dpi=120, bbox_inches="tight")
plt.show()

income_summary = df.groupby("Attrition")["MonthlyIncome"].agg(["mean", "median", "count"]).round(0)
income_summary

**Business Insights:**
1. **Leavers earn substantially less on average ($4,787 vs $6,833)** — a ~$2,000/month gap that compounds to ~$24K annually.
2. The income distribution for leavers is **shifted left** — more mass in the lower-income range.
3. Median income tells the same story — employees who leave cluster in lower pay brackets, not at the top.

**HR Implications:** Conduct pay equity reviews for roles with high attrition. Even modest adjustments for underpaid leavers-in-waiting may cost less than replacement.

---
## 6. Age Distribution

**Why:** Understanding workforce age structure informs succession planning, benefits design, and generational retention strategies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Age"], bins=20, kde=True, color="steelblue", ax=axes[0])
axes[0].set_title("Age Distribution — All Employees")
axes[0].set_xlabel("Age")

sns.histplot(data=df, x="Age", hue="Attrition", bins=20, kde=True, palette=["#2ecc71", "#e74c3c"], ax=axes[1])
axes[1].set_title("Age Distribution by Attrition Status")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_06_age_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

df["Age"].describe().round(1)

**Business Insights:**
1. Workforce age ranges **18–60** with mean ~37 — a mid-career-heavy organization.
2. The distribution is roughly normal with concentration in the **30–40 age band** — prime career-building years.
3. Leavers skew slightly younger in the overlaid histogram — early-career mobility is visible.

**HR Implications:** Benefits and career programs should address mid-career needs (growth, flexibility) while paying special attention to under-30 retention.

---
## 7. Age vs Attrition

**Why:** Age groups translate continuous age into HR-actionable segments for targeted programs.

In [ ]:
df["Age_Group"] = pd.cut(df["Age"], bins=[0, 30, 40, 50, 100], labels=["≤30", "31–40", "41–50", "50+"])

summary_7 = plot_attrition_rate(
    df, "Age_Group", "Attrition Rate by Age Group",
    save_path=IMAGES_DIR / "eda_07_age_attrition.png",
)
summary_7

**Business Insights:**
1. **Employees ≤30 have 25.9% attrition** — by far the highest age segment; early-career churn dominates.
2. **41–50 age group shows 10.6% attrition** — the most stable cohort, likely established in roles.
3. Attrition drops through mid-career then ticks up slightly at 50+ (12.6%) — possible pre-retirement transitions or role mismatch.

**HR Implications:** Invest in mentorship, rapid skill development, and clear promotion timelines for under-30 employees. This is the highest-yield age segment for retention spend.

---
## 11. Job Satisfaction vs Attrition

**Why:** Satisfaction and incentive scores are direct employee voice metrics — often more actionable than demographics.

In [ ]:
summary_11 = plot_ordinal_attrition(
    df, "JobSatisfaction", "Job Satisfaction vs Attrition",
    labels={1: '1-Low', 2: '2-Medium', 3: '3-High', 4: '4-Very High'},
    save_path=IMAGES_DIR / "eda_11_job_satisfaction.png",
)
summary_11

**Business Insights:**
1. **Job Satisfaction 1 (Low) → 22.8% attrition** vs **4 (Very High) → 11.3%** — dissatisfaction doubles flight risk.
2. The gradient is monotonic — each satisfaction step down increases attrition.
3. Even "Very High" satisfaction employees leave (11.3%) — pay and promotion still matter beyond job content.

**HR Implications:** Run stay interviews for employees rating satisfaction ≤2. Job redesign and manager coaching are cheaper than replacement.

---
## 12. Environment Satisfaction vs Attrition

**Why:** Satisfaction and incentive scores are direct employee voice metrics — often more actionable than demographics.

In [ ]:
summary_12 = plot_ordinal_attrition(
    df, "EnvironmentSatisfaction", "Environment Satisfaction vs Attrition",
    labels={1: '1-Low', 2: '2-Medium', 3: '3-High', 4: '4-Very High'},
    save_path=IMAGES_DIR / "eda_12_environment_satisfaction.png",
)
summary_12

**Business Insights:**
1. **Environment Satisfaction 1 → 25.4% attrition** — workspace and physical conditions matter more than often assumed.
2. Scores 3 and 4 cluster near **13–14%** — acceptable environments retain similarly.
3. Poor environment scores may overlap with remote-work preferences post-pandemic (though not captured in this dataset).

**HR Implications:** Audit facilities for teams with low environment scores. Hybrid work or workspace upgrades can be targeted retention tools.

---
## 13. Relationship Satisfaction vs Attrition

**Why:** Satisfaction and incentive scores are direct employee voice metrics — often more actionable than demographics.

In [ ]:
summary_13 = plot_ordinal_attrition(
    df, "RelationshipSatisfaction", "Relationship Satisfaction vs Attrition",
    labels={1: '1-Low', 2: '2-Medium', 3: '3-High', 4: '4-Very High'},
    save_path=IMAGES_DIR / "eda_13_relationship_satisfaction.png",
)
summary_13

**Business Insights:**
1. **Relationship Satisfaction 1 → 20.7% attrition** — toxic team dynamics drive exits.
2. Levels 2–4 show **14–16% attrition** — less spread than job/environment satisfaction, but low scores still hurt.
3. Manager and peer relationships are fixable levers — unlike demographics.

**HR Implications:** Train managers on team cohesion and conflict resolution. 360° feedback for leaders in high-attrition departments.

---
## 21. Stock Option Level vs Attrition

**Why:** Satisfaction and incentive scores are direct employee voice metrics — often more actionable than demographics.

In [ ]:
summary_21 = plot_ordinal_attrition(
    df, "StockOptionLevel", "Stock Option Level vs Attrition",
    labels={0: 'Level 0', 1: 'Level 1', 2: 'Level 2', 3: 'Level 3'},
    save_path=IMAGES_DIR / "eda_21_stock_options.png",
)
summary_21

**Business Insights:**
1. **No stock options (Level 0) → 24.4% attrition** — employees without equity feel less invested long-term.
2. **Levels 1–2 show ~8–10% attrition** — the strongest retention band; golden handcuffs work.
3. **Level 3 jumps to 17.6%** — senior holders may leave for better packages elsewhere despite equity.

**HR Implications:** Expand stock option eligibility below senior levels. For Level 3, focus on career meaning beyond compensation.

---
## 14. Promotion vs Attrition (Years Since Last Promotion)

**Why:** Stalled careers are a classic attrition trigger. Years since last promotion proxies promotion velocity.

In [ ]:
df["Promotion_Category"] = pd.cut(
    df["YearsSinceLastPromotion"],
    bins=[-1, 1, 3, 6, 20],
    labels=["0–1 yr", "2–3 yr", "4–6 yr", "7+ yr"],
)

summary_14 = plot_attrition_rate(
    df, "Promotion_Category", "Attrition by Years Since Last Promotion",
    xlabel="Years Since Last Promotion",
    save_path=IMAGES_DIR / "eda_14_promotion.png",
)
summary_14

**Business Insights:**
1. **4–6 years since promotion → 9.4% attrition** — the lowest rate; recent or mid-cycle promotees are more anchored.
2. **0–1 and 2–3 year bands ~17%** — employees recently promoted still leave at average rates (possibly new role mismatch).
3. **7+ years without promotion → 15.8%** — stagnation correlates with elevated, though not extreme, attrition.

**HR Implications:** Identify employees at 5+ years without promotion for career conversations. Not everyone wants promotion, but everyone wants growth visibility.

---
## 15. Years at Company

**Why:** Tenure patterns reveal when employees are most vulnerable — often early tenure (adjustment) or long tenure without change.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x="YearsAtCompany", hue="Attrition", bins=20, kde=True, palette=["#2ecc71", "#e74c3c"], ax=axes[0])
axes[0].set_title("Tenure Distribution by Attrition")

df["Tenure_Category"] = pd.cut(df["YearsAtCompany"], bins=[-1, 2, 5, 10, 100], labels=["0–2 yr", "3–5 yr", "6–10 yr", "10+ yr"])
summary_15 = plot_attrition_rate(df, "Tenure_Category", "Attrition by Tenure Category", save_path=IMAGES_DIR / "eda_15_tenure.png")

plt.tight_layout()
summary_15

**Business Insights:**
1. **0–2 years tenure → 29.8% attrition** — the onboarding and early-fit window is the highest-risk period.
2. Attrition **falls steadily with tenure**: 3–5 yr (13.8%), 6–10 yr (12.3%), 10+ yr (8.1%).
3. Classic **"honeymoon hangover"** pattern — many leave once initial excitement fades if expectations aren't met.

**HR Implications:** Strengthen 90-day and 12-month check-ins. Assign buddies/mentors in the first two years. Early tenure is where retention ROI is highest.

---
## 16. Years Since Last Promotion — Distribution

**Why:** Complements Section 14 with a continuous view of promotion timing across the workforce.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="Attrition", y="YearsSinceLastPromotion", hue="Attrition", palette=["#2ecc71", "#e74c3c"], legend=False, ax=ax)
ax.set_title("Years Since Last Promotion by Attrition Status")
ax.set_ylabel("Years Since Last Promotion")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_16_years_since_promotion.png", dpi=120, bbox_inches="tight")
plt.show()

df.groupby("Attrition")["YearsSinceLastPromotion"].agg(["mean", "median"]).round(2)

**Business Insights:**
1. **Leavers and stayers have similar median promotion timing (~2 years)** — promotion recency alone doesn't cleanly separate groups.
2. Mean years since promotion is slightly higher for stayers — long-tenured staff who stay may have accepted their career plateau.
3. Combined with Section 14, the story is about **stagnation extremes**, not average promotion cycles.

**HR Implications:** Flag outliers at 7+ years without promotion for career development plans — averages hide the stagnation tail.

---
## 17. Distance From Home

**Why:** Commute burden affects work-life balance and is a controllable factor via remote/hybrid policies.

In [ ]:
df["Distance_Band"] = pd.cut(df["DistanceFromHome"], bins=[0, 5, 10, 20, 30], labels=["1–5 mi", "6–10 mi", "11–20 mi", "21+ mi"])

summary_17 = plot_attrition_rate(
    df, "Distance_Band", "Attrition by Commute Distance",
    save_path=IMAGES_DIR / "eda_17_distance.png",
)
summary_17

**Business Insights:**
1. **21+ miles → 22.1% attrition** vs **1–5 miles → 13.8%** — longer commutes correlate with higher turnover.
2. The trend is gradual, not cliff-like — each distance band adds ~2–6 points of attrition risk.
3. Remote work (not in dataset) could mitigate this for long commuters.

**HR Implications:** Offer hybrid options for employees 15+ miles from office. Relocation assistance for critical roles if on-site is mandatory.

---
## 22. Training Times Last Year

**Why:** Training investment signals employer commitment to employee growth — too little may feel neglectful; too much may signal underperformance coaching.

In [ ]:
summary_22 = plot_ordinal_attrition(
    df, "TrainingTimesLastYear", "Attrition by Training Sessions (Last Year)",
    save_path=IMAGES_DIR / "eda_22_training.png",
)
summary_22

**Business Insights:**
1. **Zero training sessions → 27.8% attrition** — the highest rate; employees receiving no development investment leave more.
2. **6 sessions → 9.2% attrition** — highest training correlates with lowest turnover (may also reflect high-performer investment).
3. The relationship is **non-linear** — some training helps, but the zero-training penalty is the clearest signal.

**HR Implications:** Ensure every employee receives at least 2–3 training opportunities annually. Zero-training employees are a identifiable at-risk group.

---
## 23. Correlation Heatmap

**Why:** Correlations reveal multicollinearity (important for Logistic Regression) and highlight which numeric factors move together with attrition.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["Attrition_Flag", "Any_Outlier_Flag"]]

corr = df[numeric_cols + ["Attrition_Flag"]].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 7},
)
ax.set_title("Correlation Heatmap — Numeric Features vs Attrition")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_23_correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

# Top correlations with attrition
attrition_corr = corr["Attrition_Flag"].drop("Attrition_Flag").sort_values(key=abs, ascending=False)
print("Top 10 features correlated with Attrition_Flag:")
attrition_corr.head(10).round(3)

**Business Insights:**
1. **MonthlyIncome (-0.16)** and **JobLevel (-0.17)** show the strongest negative correlation with attrition — higher pay/level, lower exit probability.
2. **OverTime (encoded later), TotalWorkingYears, and YearsAtCompany** correlate with each other — tenure variables cluster; models must handle multicollinearity.
3. **StockOptionLevel (-0.14)** and **Age (-0.09)** also negatively correlate — equity and maturity reduce attrition modestly.
4. Satisfaction variables show **weak linear correlation** individually — their impact may be non-linear or interactive (trees/SHAP will capture this better).

**HR Implications:** Compensation and job level remain foundational retention levers. Satisfaction scores require segment analysis rather than linear assumptions.

---
## Phase 3 Complete — EDA Key Findings

| Rank | Driver | Evidence | Severity |
|------|--------|----------|----------|
| 1 | Overtime | 30.5% vs 10.4% | Critical |
| 2 | Low salary band | 29.3% vs ~10% high bands | Critical |
| 3 | Early tenure (0–2 yr) | 29.8% attrition | High |
| 4 | Sales Representative role | 39.8% attrition | Critical |
| 5 | Frequent business travel | 24.9% attrition | High |
| 6 | No stock options | 24.4% attrition | High |
| 7 | Low job/environment satisfaction | 22–25% attrition | High |
| 8 | Single marital status | 25.5% attrition | Moderate |
| 9 | Age ≤30 | 25.9% attrition | High |
| 10 | Zero training last year | 27.8% attrition | High |

**Next step (Phase 4):** Feature Engineering — Age Group, Tenure Category, Promotion Category (formalized for modeling).

*Awaiting confirmation to proceed to Phase 4 – Feature Engineering.*

In [ ]:
# Persist EDA-enriched dataset with temporary analysis columns for downstream phases
eda_cols = ["Age_Group", "Tenure_Category", "Promotion_Category", "Distance_Band"]
df[eda_cols].head()